# **1. Data collection**

This notebook collects the clubs and player squads from the 2026 Campeonato Brasileiro Serie A and retrieves each player's birthplace from Transfermarkt. Raw JSON responses are cached incrementally for reproducibility and safe reruns.

In [1]:
from __future__ import annotations

import gzip
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
CACHE = RAW / "transfermarkt-api"
for directory in (RAW, PROCESSED, CACHE):
    directory.mkdir(parents=True, exist_ok=True)

API_BASE = "https://tmapi.transfermarkt.technology"
WEB_BASE = "https://www.transfermarkt.com"
COMPETITION_ID = "BRA1"
SEASON_LABEL = "2026"
REFRESH_RAW = False
REQUEST_TIMEOUT = 60
REQUEST_INTERVAL = 0.8
MAX_RETRIES = 5
PLAYER_BATCH_SIZE = 100

SESSION = requests.Session()
SESSION.headers.update({
    "Accept": "application/json",
    "User-Agent": "academic-research/brazilian-league-player-birthplaces",
})
_last_request_at = 0.0


## **2. Cached Transfermarkt requests**

Every successful response is saved as compressed JSON with its source URL and retrieval time. Cached files make the pipeline resumable and preserve the exact raw snapshot used to build the processed datasets.

In [2]:
def read_cache(path: Path) -> dict[str, Any]:
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return json.load(handle)


def write_cache(path: Path, record: dict[str, Any]) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    with gzip.open(temporary, "wt", encoding="utf-8") as handle:
        json.dump(record, handle, ensure_ascii=False)
    temporary.replace(path)


def api_get(
    path: str,
    *,
    cache_name: str,
    params: list[tuple[str, Any]] | dict[str, Any] | None = None,
    refresh: bool = REFRESH_RAW,
) -> dict[str, Any] | list[Any]:
    global _last_request_at
    cache_path = CACHE / f"{cache_name}.json.gz"
    if cache_path.exists() and not refresh:
        return read_cache(cache_path)["payload"]["data"]

    url = f"{API_BASE}{path}"
    last_error: Exception | None = None
    for attempt in range(MAX_RETRIES):
        wait = REQUEST_INTERVAL - (time.monotonic() - _last_request_at)
        if wait > 0:
            time.sleep(wait)
        try:
            response = SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)
            _last_request_at = time.monotonic()
            if response.headers.get("x-amzn-waf-action") or response.status_code == 202:
                raise RuntimeError("Transfermarkt WAF challenge")
            if response.status_code == 429 or response.status_code >= 500:
                raise requests.HTTPError(f"retryable status {response.status_code}", response=response)
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict) or not payload.get("success") or "data" not in payload:
                raise ValueError("Unexpected Transfermarkt JSON response")
            record = {
                "source_url": response.url,
                "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
                "payload": payload,
            }
            write_cache(cache_path, record)
            return payload["data"]
        except (requests.RequestException, ValueError, RuntimeError) as exc:
            last_error = exc
            if attempt + 1 < MAX_RETRIES:
                time.sleep(min(30, 2 ** attempt))
    raise RuntimeError(f"Transfermarkt request failed after {MAX_RETRIES} attempts: {url}") from last_error


## **3. Season and club selection**

Transfermarkt's internal season ID is not assumed to equal the displayed year. The notebook resolves the record whose display label is `2026`, then collects exactly the participating clubs for that edition.

In [3]:
season_payload = api_get(
    f"/competition/{COMPETITION_ID}/season",
    cache_name=f"competition_{COMPETITION_ID}_seasons",
)
season_options = season_payload.get("seasons", [])
season_matches = [
    item for item in season_options
    if SEASON_LABEL in {
        str(item.get("seasonId", "")),
        str((item.get("season") or {}).get("display", "")),
        str((item.get("season") or {}).get("cyclicalName", "")),
    }
]
assert len(season_matches) == 1, f"Expected one season labelled {SEASON_LABEL}, found {len(season_matches)}"
SEASON_ID = str(season_matches[0]["seasonId"])

competition = api_get(
    f"/competition/{COMPETITION_ID}/club",
    params={"season": SEASON_ID},
    cache_name=f"competition_{COMPETITION_ID}_clubs_season_{SEASON_ID}",
)
club_ids = [str(club_id) for club_id in competition.get("clubIds", [])]
assert len(club_ids) == 20, f"Expected 20 Serie A clubs, found {len(club_ids)}"

clubs_payload = api_get(
    "/clubs",
    params=[("ids[]", club_id) for club_id in club_ids],
    cache_name=f"clubs_{COMPETITION_ID}_{SEASON_LABEL}",
)
clubs = pd.DataFrame([
    {
        "season": int(SEASON_LABEL),
        "transfermarkt_season_id": SEASON_ID,
        "competition_id": COMPETITION_ID,
        "club_id": str(club["id"]),
        "club_name": club.get("name"),
        "club_short_name": (club.get("baseDetails") or {}).get("shortName"),
        "club_url": f"{WEB_BASE}{club.get('relativeUrl')}" if club.get("relativeUrl") else None,
    }
    for club in clubs_payload
]).sort_values("club_name").reset_index(drop=True)
assert set(clubs.club_id) == set(club_ids), "One or more competition clubs are missing details"
print(f"Season displayed as {SEASON_LABEL} uses Transfermarkt season ID {SEASON_ID}")
display(clubs)


Season displayed as 2026 uses Transfermarkt season ID 2025


,season,transfermarkt_season_id,competition_id,club_id,club_name,club_short_name,club_url
0,2026,2025,BRA1,17776,Associação Chapecoense de Futebol,Chapecoense,https://www.transfermarkt.com/associacao-chape...
1,2026,2025,BRA1,537,Botafogo de Futebol e Regatas,Botafogo,https://www.transfermarkt.com/botafogo-de-fute...
2,2026,2025,BRA1,614,CR Flamengo,Flamengo,https://www.transfermarkt.com/cr-flamengo/star...
3,2026,2025,BRA1,679,Club Athletico Paranaense,Athletico-PR,https://www.transfermarkt.com/club-athletico-p...
4,2026,2025,BRA1,330,Clube Atlético Mineiro,Atlético-MG,https://www.transfermarkt.com/clube-atletico-m...
5,2026,2025,BRA1,978,Clube de Regatas Vasco da Gama,Vasco da Gama,https://www.transfermarkt.com/clube-de-regatas...
6,2026,2025,BRA1,10997,Clube do Remo (PA),Remo,https://www.transfermarkt.com/clube-do-remo-pa...
7,2026,2025,BRA1,776,Coritiba Foot Ball Club,Coritiba FC,https://www.transfermarkt.com/coritiba-foot-ba...
8,2026,2025,BRA1,609,Cruzeiro Esporte Clube,Cruzeiro,https://www.transfermarkt.com/cruzeiro-esporte...
9,2026,2025,BRA1,10010,Esporte Clube Bahia,EC Bahia,https://www.transfermarkt.com/esporte-clube-ba...


## **4. Squad collection**

The squad endpoint is collected once per club. A player may have more than one club membership if they represented multiple Serie A clubs during the edition; those memberships are preserved before creating the unique-player table.

In [4]:
membership_rows: list[dict[str, Any]] = []
for club in tqdm(clubs.itertuples(index=False), total=len(clubs), desc="Club squads"):
    squad = api_get(
        f"/club/{club.club_id}/squad",
        params={"season": SEASON_ID},
        cache_name=f"club_{club.club_id}_squad_season_{SEASON_ID}",
    )
    for member in squad.get("squad", []):
        membership_rows.append({
            "season": int(SEASON_LABEL),
            "transfermarkt_season_id": SEASON_ID,
            "competition_id": COMPETITION_ID,
            "club_id": club.club_id,
            "club_name": club.club_name,
            "player_id": str(member["playerId"]),
            "shirt_number": member.get("shirtNumber"),
            "is_captain": member.get("isCaptain"),
            "membership_type": member.get("type"),
        })

memberships = (
    pd.DataFrame(membership_rows)
    .drop_duplicates(["club_id", "player_id"])
    .sort_values(["club_name", "shirt_number", "player_id"], na_position="last")
    .reset_index(drop=True)
)
assert memberships.club_id.nunique() == 20, "A participating club has no squad data"
assert memberships.player_id.nunique() >= 400, "Unexpectedly small player universe"
print(f"{len(memberships):,} player-club memberships | {memberships.player_id.nunique():,} unique players")
display(memberships.groupby("club_name").size().rename("players").to_frame())


Club squads:   0%|          | 0/20 [00:00<?, ?it/s]

668 player-club memberships | 668 unique players


,players
club_name,
Associação Chapecoense de Futebol,38
Botafogo de Futebol e Regatas,35
CR Flamengo,28
Club Athletico Paranaense,32
Clube Atlético Mineiro,33
Clube de Regatas Vasco da Gama,32
Clube do Remo (PA),33
Coritiba Foot Ball Club,29
Cruzeiro Esporte Clube,29


## **5. Player profiles and birthplaces**

Player records are downloaded in batches of at most 100 IDs. Birth countries are decoded through Transfermarkt's country reference table, while missing birthplaces remain explicit rather than being inferred.

In [5]:
attributes = api_get("/attributes", cache_name="attributes")
country_by_id = {int(item["id"]): item for item in attributes.get("countries", [])}

player_ids = sorted(memberships.player_id.unique(), key=int)
profile_records: list[dict[str, Any]] = []
for batch_number, start in enumerate(range(0, len(player_ids), PLAYER_BATCH_SIZE), start=1):
    batch_ids = player_ids[start:start + PLAYER_BATCH_SIZE]
    batch = api_get(
        "/players",
        params=[("ids[]", player_id) for player_id in batch_ids],
        cache_name=f"players_{SEASON_LABEL}_batch_{batch_number:02d}",
    )
    profile_records.extend(batch)

collected_at_utc = datetime.now(timezone.utc).isoformat()
player_rows: list[dict[str, Any]] = []
for profile in profile_records:
    player_id = str(profile["id"])
    birth = profile.get("birthPlaceDetails") or {}
    life = profile.get("lifeDates") or {}
    nationality = profile.get("nationalityDetails") or {}
    nationality_ids = nationality.get("nationalities") or {}
    attributes = profile.get("attributes") or {}
    birth_country = country_by_id.get(int(birth.get("countryOfBirthId") or 0), {})
    primary_nationality = country_by_id.get(int(nationality_ids.get("nationalityId") or 0), {})
    secondary_nationality = country_by_id.get(int(nationality_ids.get("secondNationalityId") or 0), {})
    player_rows.append({
        "player_id": player_id,
        "player_name": profile.get("name"),
        "full_name": profile.get("displayName") or nationality.get("passportName") or profile.get("name"),
        "date_of_birth": None if life.get("isDateOfBirthUnknown") else life.get("dateOfBirth"),
        "birth_city": birth.get("placeOfBirth") or None,
        "birth_country": birth_country.get("name"),
        "birth_country_fifa_code": birth_country.get("fifaCode"),
        "primary_nationality": primary_nationality.get("name"),
        "secondary_nationality": secondary_nationality.get("name"),
        "position_group": attributes.get("positionGroupName"),
        "position": (attributes.get("position") or {}).get("name"),
        "profile_url": f"{WEB_BASE}{profile.get('relativeUrl')}" if profile.get("relativeUrl") else None,
    })

profiles = pd.DataFrame(player_rows).drop_duplicates("player_id")
membership_summary = (
    memberships.groupby("player_id", as_index=False)
    .agg(
        clubs=("club_name", lambda values: " | ".join(dict.fromkeys(values))),
        club_ids=("club_id", lambda values: " | ".join(dict.fromkeys(values))),
        club_count=("club_id", "nunique"),
    )
)
players = membership_summary.merge(profiles, on="player_id", how="left", validate="one_to_one")
players.insert(0, "season", int(SEASON_LABEL))
players.insert(1, "transfermarkt_season_id", SEASON_ID)
players.insert(2, "competition_id", COMPETITION_ID)
players["collection_status"] = "complete"
players.loc[players.player_name.isna(), "collection_status"] = "missing_profile"
players.loc[players.player_name.notna() & (players.birth_city.isna() | players.birth_country.isna()), "collection_status"] = "missing_birthplace"
players["collected_at_utc"] = collected_at_utc
players["source"] = "Transfermarkt JSON API"
players = players.sort_values(["clubs", "player_name"], na_position="last").reset_index(drop=True)
display(players.head())


,season,transfermarkt_season_id,competition_id,player_id,clubs,club_ids,club_count,player_name,full_name,date_of_birth,...,birth_country,birth_country_fifa_code,primary_nationality,secondary_nationality,position_group,position,profile_url,collection_status,collected_at_utc,source
0,2026,2025,BRA1,598974,Associação Chapecoense de Futebol,17776,1,Anderson,Anderson Silva da Paixão,1998-03-05,...,Brazil,BRA,Brazil,NaN,Goalkeeper,Goalkeeper,https://www.transfermarkt.com/anderson/profil/...,complete,2026-11-04T23:27:18.101551+00:00,Transfermarkt JSON API
1,2026,2025,BRA1,729587,Associação Chapecoense de Futebol,17776,1,Bruno Leonardo,Bruno Leonardo dos Santos Covas,1996-06-02,...,Brazil,BRA,Brazil,NaN,Defender,Centre-Back,https://www.transfermarkt.com/bruno-leonardo/p...,complete,2026-11-04T23:27:18.101551+00:00,Transfermarkt JSON API
2,2026,2025,BRA1,671292,Associação Chapecoense de Futebol,17776,1,Bruno Matias,Bruno Matias dos Santos,1999-02-24,...,Brazil,BRA,Brazil,NaN,Midfielder,Central Midfield,https://www.transfermarkt.com/bruno-matias/pro...,complete,2026-11-04T23:27:18.101551+00:00,Transfermarkt JSON API
3,2026,2025,BRA1,435163,Associação Chapecoense de Futebol,17776,1,Bruno Pacheco,Bruno de Jesus Pacheco,1991-12-08,...,Brazil,BRA,Brazil,NaN,Defender,Left-Back,https://www.transfermarkt.com/bruno-pacheco/pr...,complete,2026-11-04T23:27:18.101551+00:00,Transfermarkt JSON API
4,2026,2025,BRA1,550178,Associação Chapecoense de Futebol,17776,1,Bruno Tubarão,Bruno Nunes de Barros,1995-03-05,...,Brazil,BRA,Brazil,NaN,Striker,Right Winger,https://www.transfermarkt.com/bruno-tubarao/pr...,complete,2026-11-04T23:27:18.101551+00:00,Transfermarkt JSON API


## **6. Export and audit**

Processed tables are exported to `data/processed`. The audit checks the expected 20 clubs, profile coverage, uniqueness, and birthplace completeness.

In [6]:
profile_coverage = players.player_name.notna().mean()
birthplace_coverage = (players.birth_city.notna() & players.birth_country.notna()).mean()
audit = pd.DataFrame([
    {"metric": "clubs", "value": int(clubs.club_id.nunique())},
    {"metric": "player_club_memberships", "value": int(len(memberships))},
    {"metric": "unique_players", "value": int(len(players))},
    {"metric": "players_with_profile", "value": int(players.player_name.notna().sum())},
    {"metric": "players_with_complete_birthplace", "value": int((players.birth_city.notna() & players.birth_country.notna()).sum())},
    {"metric": "players_missing_birthplace", "value": int((players.birth_city.isna() | players.birth_country.isna()).sum())},
    {"metric": "profile_coverage_pct", "value": round(profile_coverage * 100, 2)},
    {"metric": "birthplace_coverage_pct", "value": round(birthplace_coverage * 100, 2)},
])

assert clubs.club_id.nunique() == 20, "The processed club list is incomplete"
assert players.player_id.is_unique, "The main dataset must contain one row per player"
assert profile_coverage >= 0.98, f"Profile coverage is too low: {profile_coverage:.1%}"
assert memberships.player_id.isin(players.player_id).all(), "A squad member is missing from the player universe"

clubs.to_csv(PROCESSED / f"brasileirao_{SEASON_LABEL}_clubs.csv", index=False)
memberships.to_csv(PROCESSED / f"brasileirao_{SEASON_LABEL}_squad_memberships.csv", index=False)
players.to_csv(PROCESSED / f"brasileirao_{SEASON_LABEL}_player_birthplaces.csv", index=False)
audit.to_csv(PROCESSED / f"brasileirao_{SEASON_LABEL}_collection_audit.csv", index=False)

display(audit)
display(players.collection_status.value_counts(dropna=False).rename("players").to_frame())
print(f"Processed files written to: {PROCESSED}")


,metric,value
0,clubs,20.00
1,player_club_memberships,668.00
2,unique_players,668.00
3,players_with_profile,668.00
4,players_with_complete_birthplace,661.00
5,players_missing_birthplace,7.00
6,profile_coverage_pct,100.00
7,birthplace_coverage_pct,98.95


,players
collection_status,
complete,661
missing_birthplace,7


Processed files written to: /Users/gabrielcarbinatto/Documents/GitHub/brazilian-league-2026-player-birthplaces/data/processed
